In [1]:
from sage.all import * 
from Crypto.Hash import SHAKE256

In [2]:
def int_to_4_u64(x: int):
    mask = (1 << 64) - 1
    return [
        (x >> (64 * 0)) & mask,
        (x >> (64 * 1)) & mask,
        (x >> (64 * 2)) & mask,
        (x >> (64 * 3)) & mask,
    ]

In [3]:
def u_64_chunks_to_int(chunks):
    x = 0
    for el in reversed(chunks):
        x *= 2**64
        x += el
    return x

In [4]:
def u8_to_u64(u8_list):
    assert len(u8_list) == 32
    
    result = []
    
    for i in range(0, 32, 8):
        value = 0
        for j in range(8):
            value |= u8_list[i + j] << (8 * j)  # little endian
        result.append(value)
        
    return result

In [5]:
q = 0x73eda753299d7d483339d80809a1d80553bda402fffe5bfeffffffff00000001

rounds_full_initial = 4 
rounds_full_final = 2
rounds_partial = 68

n = 4

byte_length = (rounds_full_initial + rounds_full_initial) * n + rounds_partial
byte_length *= ceil(log(q, 2**8))
byte_length

3200

In [6]:
seed = ("Neptune_" + str(q) + "_n_" + str(n) + "_rounds_full_" + str(rounds_full_initial + rounds_full_final) + "_rounds_partial_" + str(rounds_partial)).encode('ascii')
shake = SHAKE256.new()
shake.update(seed)

In [7]:
constants_raw = list(shake.read(byte_length))
constants = []

for r in range(0, rounds_full_initial):
    cnst = []
    for i in range(0, n):
        c = u_64_chunks_to_int(u8_to_u64(constants_raw[(n * r + i) * ceil(log(q, 2**8)):(n * r + i + 1) * ceil(log(q, 2**8))]))
        c = c % q
        cnst.append(int_to_4_u64(c))
    constants.append(cnst)

for r in range(0, rounds_partial):
    c = u_64_chunks_to_int(u8_to_u64(constants_raw[r * ceil(log(q, 2**8)):(r + 1) * ceil(log(q, 2**8))]))
    c = c % q
    cnst = [int_to_4_u64(c)] + (n - 1) * [4 * [0]]
    constants.append(cnst)

for r in range(0, rounds_full_final):
    cnst = []
    for i in range(0, n):
        c = u_64_chunks_to_int(u8_to_u64(constants_raw[(n * r + i) * ceil(log(q, 2**8)):(n * r + i + 1) * ceil(log(q, 2**8))]))
        c = c % q
        cnst.append(int_to_4_u64(c))
    constants.append(cnst)

In [8]:
print("[")
for cnst in constants:
    txt = "["
    for c in cnst:
        txt += str(c) + ","
    txt += "],"
    print(txt)
print("]")

[
[[10459098061806575489, 10763186790506156617, 5572775387142658013, 5111687532673810766],[12366656845460556497, 4412634818893127941, 8276614398695049702, 2081320828615347876],[9719145236211487993, 3297952562939999628, 3180387980790311277, 4188559528099826738],[18362105458242790420, 8002219859360191122, 6802251417236491332, 780488997307350801],],
[[12836875083864924791, 8421390917803026014, 11467492319742134871, 4814518759768644659],[8907719980694482196, 14121223542827703304, 4998153063538811240, 3044565554884258882],[16791809405480784945, 16859206620011810892, 9219432178401224001, 8270319353162204226],[14890490678631971867, 641327897228560948, 1144454481756489098, 7625301984424961504],],
[[13543965275947101526, 8030923262398563205, 4929810010674331815, 4423053512870570784],[8401635806934817173, 9988588153983627510, 13684823979575330098, 3611801375339080257],[2145365839988595253, 3073804418097508303, 15381500669999995518, 2628195496179753872],[14886806644125016226, 1259234646919455072,